# Avaliação das faixas experimentais no conjunto de teste

## Objetivo

Avaliar a distribuição das faixas experimentais BAIXO, MEDIO e ALTO no conjunto de teste reservado.

Serão analisados:

- Percentual de registros em cada faixa.
- Proporção de casos de baixo peso em cada faixa.
- Percentual dos casos positivos capturados por faixa.

Serão utilizados o modelo treinado e os limites
já definidos pelos percentis 50 e 85 das predições
out-of-fold do conjunto de treinamento.

Nenhum limite, hiperparâmetro ou parâmetro
do modelo será alterado após a avaliação.

In [1]:
import sys
import json
import joblib
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.model_selection import train_test_split

# Localizar a raiz do repositório
raiz = Path.cwd().resolve()

while not (raiz / ".git").exists():
    if raiz == raiz.parent:
        raise RuntimeError("Repositorio nao localizado.")

    raiz = raiz.parent

# Disponibilizar os módulos personalizados
sys.path.insert(0, str(raiz / "ml" / "src"))

from data_preparation import (
    carregar_dados,
    criar_variavel_alvo,
    separar_atributos_alvo,
    verificar_vazamento
)

print("Ambiente configurado com sucesso!")

Ambiente configurado com sucesso!


In [2]:
# Carregar a mesma base utilizada no treinamento
df = carregar_dados()

# Reproduzir o tratamento das duplicatas
df_sem_duplicados = df.drop_duplicates().copy()

# Recriar a variável-alvo
dados = criar_variavel_alvo(df_sem_duplicados)

# Separar atributos e variável-alvo
X, y = separar_atributos_alvo(dados)

# Verificar vazamento de dados
verificar_vazamento(X)

# Reproduzir a divisão original
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Validar as dimensões esperadas
assert len(df) == 120786
assert len(df_sem_duplicados) == 120781
assert len(X) == 120780
assert len(X.columns) == 10

assert len(X_treino) == 96624
assert len(X_teste) == 24156

assert int(y_treino.sum()) == 9975
assert int(y_teste.sum()) == 2494

assert len(
    X_treino.index.intersection(X_teste.index)
) == 0

# Exibir somente informações agregadas
print("Registros originais:", len(df))
print("Duplicatas removidas:", len(df) - len(df_sem_duplicados))
print("Registros validos:", len(X))
print("Atributos preditores:", len(X.columns))

print("\nConjunto de treinamento:")
print("Registros:", len(X_treino))
print("Casos positivos:", int(y_treino.sum()))

print("\nConjunto de teste:")
print("Registros:", len(X_teste))
print("Casos positivos:", int(y_teste.sum()))
print("Prevalencia (%):", round(y_teste.mean() * 100, 4))

print("\nValidacao da divisao dos dados: OK")

Verificação de vazamento: OK
Registros originais: 120786
Duplicatas removidas: 5
Registros validos: 120780
Atributos preditores: 10

Conjunto de treinamento:
Registros: 96624
Casos positivos: 9975

Conjunto de teste:
Registros: 24156
Casos positivos: 2494
Prevalencia (%): 10.3246

Validacao da divisao dos dados: OK


## Carregamento e validação do modelo treinado

Será carregado o Pipeline previamente treinado e disponibilizado nos artefatos do projeto.

Os limites das faixas experimentais serão recuperados
diretamente do arquivo metadata.json.

As métricas e a matriz de confusão do conjunto de teste serão comparadas com os resultados da avaliação original. Nenhum treinamento, ajuste de hiperparâmetros ou alteração dos limites será realizado.

In [3]:
# Localizar os artefatos
pasta_artifacts = raiz / "ml" / "artifacts"

# Carregar o modelo treinado
modelo = joblib.load(
    pasta_artifacts / "modelo_sinasc.joblib"
)

# Carregar os metadados
with open(
    pasta_artifacts / "metadata.json",
    encoding="utf-8"
) as arquivo:
    metadados = json.load(arquivo)

# Recuperar as configurações originais
configuracao = metadados["classificacao"]

threshold = configuracao["threshold"]

faixas = configuracao["faixas"]

limite_medio = faixas["MEDIO"]["min"]

limite_alto = faixas["ALTO"]["min"]

# Confirmar que os limites permanecem inalterados
assert threshold == 0.43

assert limite_medio == 0.4495018644184755

assert limite_alto == 0.5042154221007533

assert configuracao["score_calibrado"] is False

assert list(modelo.classes_) == [0, 1]

print("Modelo carregado com sucesso!")
print("Versao:", metadados["versao"])
print("Threshold:", threshold)
print("Limite BAIXO/MEDIO:", limite_medio)
print("Limite MEDIO/ALTO:", limite_alto)
print("Score calibrado:", configuracao["score_calibrado"])

print("\nValidacao dos limites originais: OK")

Modelo carregado com sucesso!
Versao: 1.0.0
Threshold: 0.43
Limite BAIXO/MEDIO: 0.4495018644184755
Limite MEDIO/ALTO: 0.5042154221007533
Score calibrado: False

Validacao dos limites originais: OK


In [4]:
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    precision_score,
    f1_score,
    fbeta_score,
    average_precision_score,
    roc_auc_score,
    confusion_matrix
)

# Gerar os scores no conjunto de teste
scores_teste = modelo.predict_proba(X_teste)[:, 1]

# Validar os scores
assert len(scores_teste) == 24156

assert np.isfinite(scores_teste).all()

assert ((scores_teste >= 0) & (scores_teste <= 1)).all()

# Aplicar o threshold original
y_pred_teste = (scores_teste >= threshold).astype(int)

# Calcular as métricas
metricas_teste = {
    "Accuracy": accuracy_score(y_teste, y_pred_teste),

    "Recall": recall_score(y_teste, y_pred_teste),

    "Precision": precision_score(y_teste, y_pred_teste),

    "F1-score": f1_score(y_teste, y_pred_teste),

    "F2-score": fbeta_score(
        y_teste, y_pred_teste, beta=2
    ),

    "Average Precision": average_precision_score(
        y_teste, scores_teste
    ),

    "ROC-AUC": roc_auc_score(
        y_teste, scores_teste
    )
}

# Exibir as métricas
display(
    pd.DataFrame(
        metricas_teste.items(),
        columns=["Metrica", "Resultado"]
    ).round(4)
)

# Reproduzir a matriz de confusão original
matriz = confusion_matrix(
    y_teste,
    y_pred_teste,
    labels=[0, 1]
)

print("Matriz de confusao:")
print(matriz)

# Comparar com os resultados originais
matriz_esperada = np.array([
    [8901, 12761],
    [649, 1845]
])

assert np.array_equal(matriz, matriz_esperada)

metricas_esperadas = {
    "Accuracy": 0.4449,
    "Recall": 0.7398,
    "Precision": 0.1263,
    "F1-score": 0.2158,
    "F2-score": 0.3753,
    "Average Precision": 0.2719,
    "ROC-AUC": 0.6407
}

for metrica, valor in metricas_esperadas.items():
    assert np.isclose(
        metricas_teste[metrica],
        valor,
        atol=0.00005,
        rtol=0
    ), f"Divergencia na metrica {metrica}"

print("\nReproducao das metricas originais: OK")
print("Quantidade de scores:", len(scores_teste))

,Metrica,Resultado
0,Accuracy,0.4449
1,Recall,0.7398
2,Precision,0.1263
3,F1-score,0.2158
4,F2-score,0.3753
5,Average Precision,0.2719
6,ROC-AUC,0.6407


Matriz de confusao:
[[ 8901 12761]
 [  649  1845]]

Reproducao das metricas originais: OK
Quantidade de scores: 24156


## Avaliação das faixas experimentais no teste

Aplicando os limites experimentais previamente definidos pelos percentis 50 e 85 dos scores out-of-fold do conjunto de treinamento. Serão avaliados os 24.156 registros do conjunto
de teste, incluindo os 2.494 casos positivos.

Para cada faixa, serão calculados:

- Quantidade de registros.
- Quantidade de casos positivos.
- Percentual de registros em relação ao teste.
- Proporção de baixo peso dentro da faixa.
- Percentual dos casos positivos capturados.

Os limites permanecerão sem altterações,
independentemente dos resultados encontrados.


In [5]:
# Aplicar os limites definidos no treinamento
faixas_teste = pd.cut(
    scores_teste,
    bins=[
        0.0,
        limite_medio,
        limite_alto,
        np.inf
    ],
    labels=["BAIXO", "MEDIO", "ALTO"],
    right=False,
    include_lowest=True
)

# Organizar os resultados
dados_faixas_teste = pd.DataFrame({
    "Faixa": faixas_teste,
    "Baixo_peso_real": y_teste.to_numpy()
})

# Consolidar os resultados por faixa
resumo_faixas_teste = dados_faixas_teste.groupby(
    "Faixa",
    observed=False
).agg(
    Registros=("Baixo_peso_real", "size"),
    Casos_positivos=("Baixo_peso_real", "sum"),
    Proporcao_positivos=("Baixo_peso_real", "mean")
).reset_index()

# Percentual de registros em cada faixa
resumo_faixas_teste["Percentual_da_base"] = (
    resumo_faixas_teste["Registros"]
    / len(y_teste) * 100
)

# Proporção de baixo peso dentro de cada faixa
resumo_faixas_teste["Proporcao_positivos"] *= 100

# Percentual dos casos positivos capturados
resumo_faixas_teste["Percentual_casos_capturados"] = (
    resumo_faixas_teste["Casos_positivos"]
    / y_teste.sum() * 100
)

# Validar os totais
assert dados_faixas_teste["Faixa"].notna().all()

assert resumo_faixas_teste["Registros"].sum() == 24156

assert resumo_faixas_teste["Casos_positivos"].sum() == 2494

assert np.isclose(
    resumo_faixas_teste["Percentual_da_base"].sum(),
    100
)

assert np.isclose(
    resumo_faixas_teste["Percentual_casos_capturados"].sum(),
    100
)

# Confirmar que os limites não foram alterados
assert threshold == 0.43
assert limite_medio == 0.4495018644184755
assert limite_alto == 0.5042154221007533

# Exibir os resultados agregados
display(
    resumo_faixas_teste[
        [
            "Faixa",
            "Registros",
            "Casos_positivos",
            "Percentual_da_base",
            "Proporcao_positivos",
            "Percentual_casos_capturados"
        ]
    ].round(4)
)

print("Total de registros:", len(y_teste))
print("Total de casos positivos:", int(y_teste.sum()))
print("Prevalencia geral (%):", round(y_teste.mean() * 100, 4))

print("\nValidacao dos totais: OK")
print("Limites originais preservados: OK")
print("Avaliacao das faixas no teste concluida!")

,Faixa,Registros,Casos_positivos,Percentual_da_base,Proporcao_positivos,Percentual_casos_capturados
0,BAIXO,12016,863,49.7433,7.1821,34.6030
1,MEDIO,8289,788,34.3145,9.5066,31.5958
2,ALTO,3851,843,15.9422,21.8904,33.8011


Total de registros: 24156
Total de casos positivos: 2494
Prevalencia geral (%): 10.3246

Validacao dos totais: OK
Limites originais preservados: OK
Avaliacao das faixas no teste concluida!


## Comparação entre treinamento OOF e teste

Os resultados obtidos no conjunto de teste serão comparados com as informações das predições out-of-fold registradas no metadata.json.

As diferenças serão calculadas em pontos
percentuais, considerando:

- Percentual de registros em cada faixa.
- Proporção de positivos em cada faixa.
- Percentual dos casos positivos capturados.

No entando, a comparação possui caráter descritivo. Ou seja,nenhum limite será recalculado ou ajustado com base nos resultados do conjunto de teste.

In [6]:
# Recuperar os resultados OOF dos metadados
resumo_oof = pd.DataFrame.from_dict(
    metadados["avaliacao_faixas_oof"],
    orient="index"
)

resumo_oof.index.name = "Faixa"

resumo_oof = resumo_oof.reset_index()

# Selecionar e padronizar os indicadores
resumo_oof = resumo_oof.rename(columns={
    "percentual_da_base": "Percentual_da_base",
    "proporcao_positivos_pct": "Proporcao_positivos",
    "percentual_casos_capturados": "Percentual_casos_capturados"
})

indicadores = [
    "Percentual_da_base",
    "Proporcao_positivos",
    "Percentual_casos_capturados"
]

# Comparar cada indicador
resultados_comparacao = []

for indicador in indicadores:

    comparacao = resumo_oof[
        ["Faixa", indicador]
    ].merge(
        resumo_faixas_teste[["Faixa", indicador]],
        on="Faixa",
        suffixes=("_OOF", "_Teste")
    )

    comparacao["Indicador"] = indicador

    comparacao = comparacao.rename(columns={
        f"{indicador}_OOF": "OOF",
        f"{indicador}_Teste": "Teste"
    })

    comparacao["Diferenca_pp"] = (
        comparacao["Teste"] - comparacao["OOF"]
    )

    resultados_comparacao.append(comparacao)

# Consolidar as comparações
df_comparacao = pd.concat(
    resultados_comparacao,
    ignore_index=True
)

display(
    df_comparacao[
        [
            "Indicador",
            "Faixa",
            "OOF",
            "Teste",
            "Diferenca_pp"
        ]
    ].round(4)
)

print("Comparacao entre OOF e teste concluida!")
print("Nenhum limite foi alterado.")

,Indicador,Faixa,OOF,Teste,Diferenca_pp
0,Percentual_da_base,BAIXO,49.9472,49.7433,-0.2039
1,Percentual_da_base,MEDIO,35.0524,34.3145,-0.7379
2,Percentual_da_base,ALTO,15.0004,15.9422,0.9418
3,Proporcao_positivos,BAIXO,7.1403,7.1821,0.0417
4,Proporcao_positivos,MEDIO,9.5751,9.5066,-0.0686
5,Proporcao_positivos,ALTO,22.6715,21.8904,-0.7810
6,Percentual_casos_capturados,BAIXO,34.5464,34.6030,0.0567
7,Percentual_casos_capturados,MEDIO,32.5113,31.5958,-0.9154
8,Percentual_casos_capturados,ALTO,32.9424,33.8011,0.8588


Comparacao entre OOF e teste concluida!
Nenhum limite foi alterado.


## Conclusão da avaliação das faixas no teste

A avaliação foi realizada com os 24.156 registros do conjunto de teste original, incluindo 2.494 casos positivos de baixo peso, os limites definidos anteriormente pelos percentis
50 e 85 foram preservados integralmente.

A faixa ALTO concentrou 15,94% dos registros, apresentando frequência observada de baixo peso de 21,89% e concentrando 33,80% dos casos positivos. Na comparação entre as predições out-of-fold e o conjunto de teste, todos os indicadores apresentaram diferenças absolutas inferiores a um ponto percentual.

Os resultados indicam consistência descritiva das faixas dentro da base estudada. Entretanto, 66,20% dos casos positivos do teste permaneceram fora da faixa ALTO.

Nenhum treinamento, ajuste de hiperparâmetros
ou redefinição dos limites foi realizado.